# AI-Generated Code Detection and Similarity Analysis — Manual ChatGPT Edition
No paid API needed. Section 2 imports `.py` programs you generate manually in ChatGPT. Results from small manually assembled datasets are exploratory and must not be described as production-ready authorship detection.


In [ ]:
!pip -q install scikit-learn xgboost pandas tqdm joblib

In [ ]:
import os, re, ast, io, json, hashlib, random, time, tokenize, subprocess, builtins
from pathlib import Path
import numpy as np, pandas as pd
from tqdm.auto import tqdm

random.seed(42); np.random.seed(42)
WORK = Path("/content/codedetect"); WORK.mkdir(parents=True, exist_ok=True)

# ---- settings you can change ----
AI_MODELS = ["chatgpt-manual"]   # manually created, no API
MAX_PROBLEMS = 20  # start with 20; increase after successful pilot          # human files / tasks to use
MIN_LINES, MAX_LINES = 8, 120
STRIP_DOCSTRINGS = True     # removes docstrings + doctests from human code (see note below)
STYLE_VARIANTS = ["", "Write it the way a second-year CS student would in a lab record."]



## 1. Human code
Source: **TheAlgorithms/Python** (MIT licensed). Each file is one task. You can add your own / friends' lab programs (with permission) in `human_own/`, named like `reverse_a_string.py`.

> **Confound to mention in your report:** TheAlgorithms files have heavy docstrings and doctests, and AI code has them too, so a model could learn "has docstring" instead of "is AI". `STRIP_DOCSTRINGS=True` removes this shortcut from the human side; the AI side is left as generated, so also check the ablation of dropping docstring features.

In [ ]:
REPO = WORK / "TheAlgorithms"
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/TheAlgorithms/Python.git", str(REPO)], check=True)

DIRS = ["maths", "strings", "sorts", "searches", "data_structures", "dynamic_programming",
        "graphs", "greedy_methods", "backtracking", "bit_manipulation", "conversions",
        "ciphers", "matrix", "other", "divide_and_conquer"]

MAIN_DOCTEST = re.compile(r'if __name__ == "__main__":\s*\n\s+(import doctest\s*\n\s+)?doctest\.testmod\(\)[^\n]*\n?')

def strip_docs(src):
    src = MAIN_DOCTEST.sub("", src)
    try:
        tree = ast.parse(src)
    except SyntaxError:
        return None
    kill = set()
    for node in ast.walk(tree):
        if isinstance(node, (ast.Module, ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            b = node.body
            if b and isinstance(b[0], ast.Expr) and isinstance(getattr(b[0], "value", None), ast.Constant) \
               and isinstance(b[0].value.value, str):
                kill.update(range(b[0].lineno, b[0].end_lineno + 1))
    lines = [l for i, l in enumerate(src.splitlines(), 1) if i not in kill and "doctest" not in l]
    return "\n".join(lines).strip() + "\n"

def valid(src):
    try:
        ast.parse(src); return True
    except Exception:
        return False

human = []
for d in DIRS:
    for p in sorted((REPO / d).rglob("*.py")):
        if p.name.startswith(("__", "test_")) or "test" in p.stem:
            continue
        src = p.read_text(errors="ignore")
        if STRIP_DOCSTRINGS:
            src = strip_docs(src)
        if not src or not valid(src):
            continue
        n = len([l for l in src.splitlines() if l.strip()])
        if not (MIN_LINES <= n <= MAX_LINES):
            continue
        human.append(dict(problem_id=f"{d}/{p.stem}", area=d.replace("_", " "),
                          title=p.stem.replace("_", " "), code=src, label=0,
                          source="TheAlgorithms", model="human"))

# your own lab programs (optional): /content/codedetect/human_own/<task_name>.py
OWN = WORK / "human_own"; OWN.mkdir(exist_ok=True)
for p in sorted(OWN.glob("*.py")):
    src = p.read_text(errors="ignore")
    if valid(src):
        human.append(dict(problem_id=f"own/{p.stem}", area="lab program",
                          title=p.stem.replace("_", " "), code=src, label=0,
                          source="own", model="human"))

random.shuffle(human)
human = human[:MAX_PROBLEMS]
print("human samples:", len(human))

## 2. Manually generated ChatGPT code (no API key or payment)

Run this cell to generate a CSV task list and a prompt. Download `manual_tasks.csv`, ask ChatGPT for each task's Python code, and save the code in `/content/codedetect/ai_manual/` using the **exact** filename from the CSV. Use the Colab Files pane to create/upload the folder, or upload a ZIP of `.py` files and extract it using the optional helper below. Do not use the human repository source code as the prompt. **Start with 20 tasks**, and save at least 10 valid files before training. This is a small demonstration dataset, not a reliable AI-detector benchmark.


In [ ]:
from IPython.display import display
import pandas as pd, json, re, zipfile
MANUAL = WORK / "ai_manual"
MANUAL.mkdir(parents=True, exist_ok=True)

# Stable file naming so each AI file matches its corresponding human programming task.
# If you change MAX_PROBLEMS or rerun Section 1, the selected task list may change.
tasks = []
for idx, h in enumerate(human, 1):
    filename = f"task_{idx:04d}.py"
    tasks.append({"filename": filename, "problem_id": h["problem_id"],
                  "title": h["title"], "area": h["area"]})
task_table = pd.DataFrame(tasks)
task_table.to_csv(WORK / "manual_tasks.csv", index=False)
print("Tasks to generate:", len(task_table))
display(task_table.head(20))
print("\nPrompt template (replace the bracketed items with one row):\n")
print("Write a complete standalone Python program for the task: [TITLE]. Topic: [AREA]. "
      "Use Python only. Return only the Python code, with no markdown fences or explanation. "
      "Do not ask for or copy a reference implementation.")
print("\nSave each answer using its matching filename in", MANUAL)

# OPTIONAL: upload a zip of your saved .py programs; extracts to ai_manual/.
# Uncomment these lines only when you are ready to upload the zip.
# from google.colab import files
# uploaded = files.upload()
# for name, contents in uploaded.items():
#     if name.lower().endswith('.zip'):
#         with zipfile.ZipFile(__import__('io').BytesIO(contents)) as archive:
#             for member in archive.namelist():
#                 if member.endswith('.py') and not member.startswith('__MACOSX'):
#                     target = MANUAL / __import__('pathlib').Path(member).name
#                     target.write_bytes(archive.read(member))

# Rebuild 'done' from files on each run; missing/invalid files are reported.
done = {}
missing, invalid = [], []
for t in tasks:
    path = MANUAL / t["filename"]
    if not path.exists():
        missing.append(t["filename"])
        continue
    code = path.read_text(encoding="utf-8-sig", errors="replace").strip() + "\n"
    if not valid(code):
        invalid.append(t["filename"])
        continue
    rec = dict(problem_id=t["problem_id"], area=t["area"], title=t["title"],
               code=code, label=1, source="manual_chatgpt", model="chatgpt-manual", style="manual")
    done[(t["problem_id"], "chatgpt-manual")] = rec
print(f"Valid AI samples: {len(done)} / {len(tasks)}")
print("Missing files (first 10):", missing[:10])
print("Invalid Python files:", invalid[:10])
if len(done) < 10:
    raise ValueError("Add at least 10 valid ChatGPT-generated .py files to ai_manual/ and rerun this cell.")


## 3. Clean, dedupe, validate, balance, split by problem

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

def norm_hash(src):
    return hashlib.md5(re.sub(r"\s+", "", src).encode()).hexdigest()

full = pd.DataFrame(human + list(done.values()))
full["n_lines"] = full.code.map(lambda s: len([l for l in s.splitlines() if l.strip()]))
full = full[full.code.map(valid) & full.n_lines.between(MIN_LINES // 2, MAX_LINES * 2)]
full["h"] = full.code.map(norm_hash)
full = full.drop_duplicates("h").drop(columns="h").reset_index(drop=True)

# balanced set: all human + ONE random AI sample per problem. Extra AI samples are kept for generalisation tests.
ai = full[full.label == 1]
ai_bal = ai.sample(frac=1, random_state=42).drop_duplicates("problem_id")   # one random AI sample per problem
hu = full[full.label == 0]
common = set(hu.problem_id) & set(ai_bal.problem_id)
data = pd.concat([hu[hu.problem_id.isin(common)], ai_bal[ai_bal.problem_id.isin(common)]]).reset_index(drop=True)
extra_ai = ai.drop(ai_bal.index, errors="ignore")

gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
tr_idx, rest_idx = next(gss.split(data, groups=data.problem_id))
rest = data.iloc[rest_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
va_i, te_i = next(gss2.split(rest, groups=rest.problem_id))
train, val, test = data.iloc[tr_idx], rest.iloc[va_i], rest.iloc[te_i]

for n, d in [("train", train), ("val", val), ("test", test)]:
    print(n, len(d), d.label.value_counts().to_dict())
data.to_csv(WORK / "dataset.csv", index=False)
extra_ai.to_csv(WORK / "extra_ai.csv", index=False)
print("saved ->", WORK / "dataset.csv")

## 4. Features: stylometric + AST node counts

In [ ]:
from collections import Counter
NEST = (ast.If, ast.For, ast.While, ast.Try, ast.With, ast.FunctionDef, ast.ClassDef)
SNAKE = re.compile(r"^[a-z_][a-z0-9_]*$")

def depth(node, d=0):
    m = d
    for c in ast.iter_child_nodes(node):
        m = max(m, depth(c, d + 1 if isinstance(c, NEST) else d))
    return m

def features(src):
    lines = src.splitlines() or [""]
    nb = [l for l in lines if l.strip()]
    lens = [len(l) for l in nb] or [0]
    try:
        toks = list(tokenize.generate_tokens(io.StringIO(src).readline))
    except Exception:
        toks = []
    comments = [t for t in toks if t.type == tokenize.COMMENT]
    tree = ast.parse(src)
    names = []
    for n in ast.walk(tree):
        if isinstance(n, ast.Name): names.append(n.id)
        elif isinstance(n, (ast.FunctionDef, ast.ClassDef)): names.append(n.name)
        elif isinstance(n, ast.arg): names.append(n.arg)
    nodes = [type(n).__name__ for n in ast.walk(tree)]
    total = max(len(nodes), 1)
    f = {
        "n_lines": len(nb),
        "avg_line_len": float(np.mean(lens)), "max_line_len": max(lens), "std_line_len": float(np.std(lens)),
        "blank_ratio": 1 - len(nb) / max(len(lines), 1),
        "comment_density": len(comments) / max(len(nb), 1),
        "inline_comment_ratio": sum(1 for t in comments if t.line.strip() and not t.line.strip().startswith("#")) / max(len(comments), 1),
        "avg_ident_len": float(np.mean([len(x) for x in names])) if names else 0,
        "unique_ident_ratio": len(set(names)) / max(len(names), 1),
        "snake_ratio": float(np.mean([bool(SNAKE.match(x)) for x in names])) if names else 0,
        "single_char_ratio": float(np.mean([len(x) == 1 for x in names])) if names else 0,
        "n_functions": nodes.count("FunctionDef"),
        "max_nesting": depth(tree),
        "trailing_ws": sum(1 for l in lines if l != l.rstrip()),
        "tab_lines": sum(1 for l in lines if l.startswith("\t")),
        "has_type_hints": int(any(isinstance(n, ast.arg) and n.annotation for n in ast.walk(tree)) or
                              any(isinstance(n, ast.FunctionDef) and n.returns for n in ast.walk(tree))),
        "n_docstrings": sum(1 for n in ast.walk(tree) if isinstance(n, (ast.Module, ast.FunctionDef, ast.ClassDef)) and ast.get_docstring(n)),
        "uses_main_guard": int("__name__" in names),
        "n_fstrings": nodes.count("JoinedStr"),
        "n_prints": sum(1 for n in ast.walk(tree) if isinstance(n, ast.Call) and getattr(n.func, "id", "") == "print"),
    }
    for k, v in Counter(nodes).items():
        f["ast_" + k] = v / total
    return f

def featurize(df, cols=None):
    X = pd.DataFrame([features(s) for s in df.code]).fillna(0)
    if cols is not None:
        X = X.reindex(columns=cols, fill_value=0)
    return X

Xtr = featurize(train); COLS = list(Xtr.columns)
Xva, Xte = featurize(val, COLS), featurize(test, COLS)
ytr, yva, yte = train.label.values, val.label.values, test.label.values
print(Xtr.shape)

## 5. Classical models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from xgboost import XGBClassifier

models = {
    "LogReg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "RandomForest": RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8,
                             colsample_bytree=0.8, eval_metric="logloss", random_state=42),
}

def report(y, p, prob):
    return dict(acc=accuracy_score(y, p), prec=precision_score(y, p), rec=recall_score(y, p),
                f1=f1_score(y, p), auc=roc_auc_score(y, prob))

rows = []
for name, m in models.items():
    m.fit(Xtr, ytr)
    prob = m.predict_proba(Xte)[:, 1]
    r = report(yte, (prob > 0.5).astype(int), prob); r["model"] = name
    rows.append(r)
    print(name, "confusion matrix (rows=true, cols=pred):\n", confusion_matrix(yte, (prob > 0.5).astype(int)))
pd.DataFrame(rows).set_index("model").round(3)

## 6. Robustness tests
Same test set, but every program is transformed first. A drop in score shows what the model is really relying on.

In [ ]:
BUILTINS = set(dir(builtins))

class Renamer(ast.NodeTransformer):
    def __init__(self, keep):
        self.keep, self.map = keep, {}
    def _new(self, old):
        if old in self.keep or old in BUILTINS or old.startswith("__"):
            return old
        return self.map.setdefault(old, f"v{len(self.map)}")
    def visit_Name(self, n): n.id = self._new(n.id); return n
    def visit_arg(self, n): n.arg = self._new(n.arg); self.generic_visit(n); return n
    def visit_FunctionDef(self, n): n.name = self._new(n.name); self.generic_visit(n); return n

def t_reformat(src):            # strips comments, normalises whitespace/quotes
    return ast.unparse(ast.parse(src))

def t_rename(src):              # rename variables/functions/args + reformat
    tree = ast.parse(src)
    keep = set()
    for n in ast.walk(tree):
        if isinstance(n, ast.Import): keep |= {(a.asname or a.name).split(".")[0] for a in n.names}
        if isinstance(n, ast.ImportFrom): keep |= {a.asname or a.name for a in n.names}
    tree = Renamer(keep).visit(tree)
    return ast.unparse(tree)

def t_no_comments(src):
    out = []
    for t in tokenize.generate_tokens(io.StringIO(src).readline):
        pass
    lines = []
    for l in src.splitlines():
        if l.strip().startswith("#"): continue
        lines.append(re.sub(r"\s+#.*$", "", l) if "#" in l and not re.search(r"['\"].*#.*['\"]", l) else l)
    return "\n".join(lines) + "\n"

TRANSFORMS = {"original": lambda s: s, "no_comments": t_no_comments,
              "reformatted": t_reformat, "renamed+reformatted": t_rename}

def safe(fn, s):
    try:
        out = fn(s); ast.parse(out); return out
    except Exception:
        return s

best = "XGBoost"
rob = []
for tname, fn in TRANSFORMS.items():
    tdf = test.copy(); tdf["code"] = tdf.code.map(lambda s: safe(fn, s))
    Xt = featurize(tdf, COLS)
    for mname, m in models.items():
        prob = m.predict_proba(Xt)[:, 1]
        r = report(yte, (prob > 0.5).astype(int), prob); r.update(transform=tname, model=mname)
        rob.append(r)
rob = pd.DataFrame(rob)
rob.pivot(index="transform", columns="model", values="f1").round(3)

In [ ]:
# Cross-model generalisation is not applicable with only one manual ChatGPT source.
# To evaluate this, collect a separate, labeled dataset from a genuinely different generator.
print("Cross-model generalisation skipped: only manual ChatGPT code is available.")


## 7. Similarity checker (winnowing)

In [ ]:
def code_tokens(src):
    out = []
    for t in tokenize.generate_tokens(io.StringIO(src).readline):
        if t.type == tokenize.NAME:
            import keyword
            out.append(t.string if keyword.iskeyword(t.string) else "ID")
        elif t.type == tokenize.NUMBER: out.append("NUM")
        elif t.type == tokenize.STRING: out.append("STR")
        elif t.type == tokenize.OP: out.append(t.string)
    return out

def fingerprints(src, k=5, w=4):
    toks = code_tokens(src)
    if len(toks) < k: return set()
    hashes = [int(hashlib.md5(" ".join(toks[i:i + k]).encode()).hexdigest()[:8], 16) for i in range(len(toks) - k + 1)]
    fp = set()
    for i in range(max(len(hashes) - w + 1, 1)):
        win = hashes[i:i + w]
        fp.add(min(win))                       # winnowing: keep the minimum hash of each window
    return fp

def similarity(a, b):
    fa, fb = fingerprints(a), fingerprints(b)
    return len(fa & fb) / max(min(len(fa), len(fb)), 1)    # containment of the smaller one

# demo: most similar pairs inside the test split
codes = list(zip(test.problem_id, test.label, test.code))
fps = [fingerprints(c) for _, _, c in codes]
pairs = []
for i in range(len(codes)):
    for j in range(i + 1, len(codes)):
        s = len(fps[i] & fps[j]) / max(min(len(fps[i]), len(fps[j])), 1)
        pairs.append((s, codes[i][0], codes[j][0], codes[i][1], codes[j][1]))
pd.DataFrame(sorted(pairs, reverse=True)[:10], columns=["similarity", "A", "B", "A_is_ai", "B_is_ai"])

## 8. CodeBERT fine-tuning
**Needs a GPU.** Set Runtime -> Change runtime type -> T4 GPU *before you start the notebook* (changing it later restarts the session and loses variables). This section uses the same train/val/test split as the classical models.

In [ ]:
!pip -q install transformers accelerate

In [ ]:
import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)

CKPT = "microsoft/codebert-base"      # or "microsoft/unixcoder-base"
MAXLEN = 384
tok = AutoTokenizer.from_pretrained(CKPT)

class CodeDS(torch.utils.data.Dataset):
    def __init__(self, codes, labels):
        self.enc = tok(list(codes), truncation=True, max_length=MAXLEN)
        self.labels = [int(x) for x in labels]
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        d = {k: v[i] for k, v in self.enc.items()}
        d["labels"] = self.labels[i]
        return d

def compute_metrics(p):
    logits, y = p
    pred = logits.argmax(-1)
    return {"acc": accuracy_score(y, pred), "f1": f1_score(y, pred)}

bert = AutoModelForSequenceClassification.from_pretrained(CKPT, num_labels=2)
args = TrainingArguments(
    output_dir=str(WORK / "bert_out"), num_train_epochs=4, per_device_train_batch_size=8,
    per_device_eval_batch_size=16, learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="f1", save_total_limit=1, fp16=torch.cuda.is_available(),
    report_to="none", seed=42)
trainer = Trainer(model=bert, args=args, train_dataset=CodeDS(train.code, train.label),
                  eval_dataset=CodeDS(val.code, val.label), data_collator=DataCollatorWithPadding(tok),
                  compute_metrics=compute_metrics)
trainer.train()
trainer.save_model(str(WORK / "codebert")); tok.save_pretrained(str(WORK / "codebert"))

In [ ]:
from scipy.special import softmax

def bert_prob(codes):
    out = trainer.predict(CodeDS(codes, [0] * len(codes)))
    return softmax(out.predictions, axis=-1)[:, 1]

brow = []
for tname, fn in TRANSFORMS.items():
    codes = [safe(fn, s) for s in test.code]
    prob = bert_prob(codes)
    r = report(yte, (prob > 0.5).astype(int), prob); r.update(transform=tname, model="CodeBERT")
    brow.append(r)
rob_all = pd.concat([rob, pd.DataFrame(brow)])
print("F1 by transform:")
display(rob_all.pivot(index="transform", columns="model", values="f1").round(3))
print("ROC-AUC by transform:")
display(rob_all.pivot(index="transform", columns="model", values="auc").round(3))
rob_all.to_csv(WORK / "robustness_results.csv", index=False)

## 9. Error analysis
Look at what the best model gets wrong. These examples are good material for the report and viva.

In [ ]:
prob = models["XGBoost"].predict_proba(Xte)[:, 1]
err = test.assign(p_ai=prob, pred=(prob > 0.5).astype(int))
fp_ = err[(err.label == 0) & (err.pred == 1)].sort_values("p_ai", ascending=False)   # humans flagged as AI
fn_ = err[(err.label == 1) & (err.pred == 0)].sort_values("p_ai")                    # AI that slipped through
print(f"false positives (human flagged as AI): {len(fp_)} | false negatives (AI missed): {len(fn_)}")
for name, d in [("HUMAN flagged as AI", fp_), ("AI missed", fn_)]:
    if len(d):
        r = d.iloc[0]
        print(f"\n===== {name}: {r.problem_id} (p_ai={r.p_ai:.2f}) =====\n{r.code[:900]}")

## 10. Export artifacts for the Streamlit app

In [ ]:
import joblib, shutil
ART = WORK / "artifacts"; ART.mkdir(exist_ok=True)
joblib.dump({"xgb": models["XGBoost"], "rf": models["RandomForest"], "cols": COLS}, ART / "model.joblib")
pd.concat([train, val])[["problem_id", "label", "model", "code"]].to_csv(ART / "reference.csv", index=False)
shutil.make_archive("/content/artifacts", "zip", ART)
if (WORK / "codebert").exists():
    shutil.make_archive("/content/codebert", "zip", WORK / "codebert")
try:
    from google.colab import files
    files.download("/content/artifacts.zip")
    if (WORK / "codebert").exists():
        files.download("/content/codebert.zip")      # optional: ~450 MB, only needed if you want CodeBERT in the app
except Exception as e:
    print("Download manually from the Colab file browser:", e)

## Done
Download `artifacts.zip` (and optionally `codebert.zip`), unzip next to `app.py` and `core.py`, then run `streamlit run app.py` (see README).

**Report limitations:** detectors degrade on edited code and newer LLMs, can flag clean textbook-style students, and the score is a signal for a teacher to review, not proof.